# Appendix parameter sweeps

Select `GROUP`: `matched` for in-distribution sweeps, `energies` for independent training/test differing-site signals, or `means` for signed reference contributions. Run all cells to load the included optimized curves and barrier evidence, redraw the figure and save its PDF. Set `RECOMPUTE=True` to rerun the selected numerical stages.

The search includes exact zero ridge. Risks omit the independent test-noise floor and are divided by each curve's own $R_\infty$. Dashed connectors use stored two-endpoint barrier evidence; plotting does not classify transitions or evaluate risk.


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import inspect
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from copy import deepcopy

# Use only this distribution's local utilities, caches and output directories.
BUNDLE_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p/'dms'/'config.py').is_file()
                    and (p/'notebooks'/'04_parameter_sweeps.ipynb').is_file()), None)
if BUNDLE_ROOT is None:
    raise RuntimeError('Start the kernel in the reproduction folder or its notebooks/ directory.')
foreign = [name for name, module in sys.modules.items()
           if (name == 'dms' or name.startswith('dms.'))
           and getattr(module, '__file__', None)
           and not Path(module.__file__).resolve().is_relative_to(BUNDLE_ROOT/'dms')]
if foreign:
    raise RuntimeError('Restart the kernel: another checkout of dms is already imported.')
sys.path.insert(0, str(BUNDLE_ROOT))
PROJECT_ROOT = BUNDLE_ROOT  # retained renderer name; always local to this bundle
OUTPUT_DIR = BUNDLE_ROOT/'output'
importlib.invalidate_caches()
for module_name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory', 'dms.optimization', 'dms.ridge_diagnostics', 'dms.exploration', 'dms.differing_signal_sweeps', 'dms.result_cache', 'dms.transition_diagnostics', 'dms.sweep_transitions'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)
from dms.theory import DMSTTheory
from dms.stieltjes import project_stieltjes_evaluator
from dms.exploration import SweepParameters, parameter_sweep_cases, optimize_parameter_sweep
from dms.differing_signal_sweeps import differing_signal_sweep_cases
from dms.result_cache import ResultCache
from dms.sweep_transitions import cached_sweep_transitions, sweep_transition_signature
plt.rcParams.update({'axes.labelsize':11, 'axes.labelweight':'bold',
                     'xtick.labelsize':11, 'ytick.labelsize':11, 'legend.fontsize':11})
theory = DMSTTheory(project_stieltjes_evaluator())
results, TRANSITIONS, APPENDIX_FIGURES = {}, {}, {}

from tools.portable_results import load_results, assert_settings_match

# False: reproduce from the included arrays. True: calculate from the inputs below.
RECOMPUTE = False


In [ ]:
# One group at a time; changing this selector never changes the numerical grids.
GROUP = 'matched'
BASE = SweepParameters(
    q=20,
    sigma=1.0,
    G2=100.0,
    f=0.1,
    ftilde=0.2,
    eta=0.2,
    F=0.4,
    FTest=0.4,
    Fneq=0.2,
    rhoneq=0.25,  # Cneq = rhoneq * sqrt(Fneq * (FTest-F+Fneq))
    m=0.05,
    mneq=0.1,
    mneqtilde=0.1,
)
MEAN_CONVENTION = "normalized"
REFERENCE_L = 100

REGIMES = {
    "matched": {
        "fixed": dict(eta=0.0, Fneq=0.0, rhoneq=0.0, m=0.0, mneq=0.0, mneqtilde=0.0),
        "ties": {"ftilde": "f", "FTest": "F"},
        "caption": r"Matched: $\tilde f=f$, $F_{Test}=F$, $\eta=0$; means and mismatch moments zero",
    },
    "general": {"fixed": {}, "ties": {}, "caption": "General case: all other independent parameters at the shared baseline"},
}
SWEEPS = {
    "G2": {"parameter": "G2", "values": [25.0, 50.0, 100.0, 200.0],
           "regime": "matched", "symbol": r"G^2", "title": r"Signal strength $G^2$"},
    "F": {"parameter": "F", "values": [0.05, 0.1, 0.2, 0.4, 0.8],
          "regime": "matched", "symbol": r"F", "title": r"Matched alignment $F=F_{Test}$"},
    "f_common": {"parameter": "f", "values": [0.05, 0.1, 0.2, 0.4],
                 "regime": "matched", "symbol": r"f=\tilde f", "title": r"Common mutation fraction $f=\tilde f$"},
    "m": {"parameter": "m", "values": [-0.2,-0.1, 0.0, 0.1, 0.2],
          "regime": "general", "symbol": r"m", "title": r"Total mean $m$"},
    "mneq": {"parameter": "mneq", "values": [-0.2,-0.1, 0.0, 0.1, 0.2],
             "regime": "general", "symbol": 'm_{\\neq}', "title": 'Training mismatch mean $m_{\\neq}$'},
    "mneqtilde": {"parameter": "mneqtilde", "values": [-0.2,-0.1, 0.0, 0.1, 0.2],
                  "regime": "general", "symbol": '\\tilde m_{\\neq}', "title": 'Test mismatch mean $\\tilde m_{\\neq}$'}
}

R_MIN = 1e-2
R_MAX = 5*1e2
N_R = 101
R_VALUES = np.geomspace(R_MIN, R_MAX, N_R)
SEARCH = dict(method="nonnegative", lambda_max=1e3, n_grid=161)

# Independent differing-site signal paths; FTest=F-Fneq+Fneqtilde.
# Independent differing-site signals: same values in both columns.
ENERGY_VALUES = [0.1, 0.2, 0.3, 0.39]
ENERGY_FIXED_FNEQ = BASE.Fneq
ENERGY_FIXED_FNEQTILDE = BASE.FTest - BASE.F + BASE.Fneq
ENERGY_TRANSITION_OPTIONS = dict(n_probe=65, barrier_rtol=1e-8)
ENERGY_KEYS = ['Fneq_independent', 'Fneqtilde_independent']
# F remains BASE.F; FTest = F - Fneq + Fneqtilde in both paths.
SWEEPS.update({
    ENERGY_KEYS[0]: dict(parameter='Fneq', values=list(ENERGY_VALUES), regime='general',
        path='differing_signals', fixed_other=ENERGY_FIXED_FNEQTILDE,
        symbol=r'F_{\neq}', title=r'Training differing-site signal $F_{\neq}$'),
    ENERGY_KEYS[1]: dict(parameter='Fneqtilde', values=list(ENERGY_VALUES), regime='general',
        path='differing_signals', fixed_other=ENERGY_FIXED_FNEQ,
        symbol=r'\tilde F_{\neq}', title=r'Test differing-site signal $\tilde F_{\neq}$'),
})

APPENDIX_GROUPS = {
    'matched': ['G2', 'F', 'f_common'],
    'energies': ENERGY_KEYS,
    'means': ['m', 'mneq', 'mneqtilde'],
}
if GROUP not in APPENDIX_GROUPS:
    raise ValueError(f'Choose GROUP from {tuple(APPENDIX_GROUPS)}')

# Full scientific inputs and source hashes enter persistent cache identities.
CACHE_OPTIONS = dict(directory=BUNDLE_ROOT/'.cache'/'parameter_sweeps', enabled=True, verbose=True)
FORCE_THEORY = False
FORCE_DIAGNOSTICS = False
TRANSITION_OPTIONS = dict(n_probe=65, barrier_rtol=1e-8)
STUDY_TRANSITION_OPTIONS = {key:dict(ENERGY_TRANSITION_OPTIONS) for key in ENERGY_KEYS}


The in-distribution group sets $\eta=0$, zero signed means, $\tilde f=f$ and $\tilde F=F$. Energy and mean groups use the displayed distribution-shift baseline. Normalized means are calibrated at $L_{\rm ref}=100$; changing projection energies at fixed normalized means also changes the corresponding raw moments.

`SEARCH['lambda_max']` is a finite cap, not an infinity certificate. Numerical inputs are checked against the included arrays. Changing presentation controls does not require a numerical calculation.


In [ ]:
THEORY_SOURCES = [BUNDLE_ROOT/'dms'/(name+'.py') for name in (
    'config','stieltjes','theory','simulation','optimization','ridge_diagnostics',
    'exploration','differing_signal_sweeps')]
TRANSITION_SOURCES = [BUNDLE_ROOT/'dms'/(name+'.py') for name in (
    'sweep_transitions','transition_diagnostics','ridge_diagnostics','stieltjes','theory','config')]


def study_inputs(key):
    spec = SWEEPS[key]
    return dict(baseline=asdict(BASE), study=spec, regime=REGIMES[spec['regime']],
                mean_convention=MEAN_CONVENTION, reference_L=REFERENCE_L,
                r_values=R_VALUES, search=SEARCH)


def prepare_study_cases(key):
    spec = SWEEPS[key]
    if spec.get('path') == 'differing_signals':
        return differing_signal_sweep_cases(BASE, spec['parameter'], spec['values'],
            fixed_other=spec['fixed_other'], mean_convention=MEAN_CONVENTION, reference_L=REFERENCE_L)
    regime = REGIMES[spec['regime']]
    return parameter_sweep_cases(BASE, spec['parameter'], spec['values'],
        fixed=regime['fixed'], ties=regime['ties'],
        mean_convention=MEAN_CONVENTION, reference_L=REFERENCE_L)


def published_sweep_settings():
    keys = APPENDIX_GROUPS[GROUP]
    return dict(studies={key: study_inputs(key) for key in keys},
                transition_options=TRANSITION_OPTIONS,
                study_transition_options={key: STUDY_TRANSITION_OPTIONS.get(key, {}) for key in keys})


def published_sweeps():
    data, metadata = load_results(BUNDLE_ROOT/'data'/f'sweeps_{GROUP}', restore_records=True)
    assert_settings_match(metadata['settings'], published_sweep_settings())
    return data


def load_study(key, *, compute=False, force=False, prepared_cases=None):
    if not RECOMPUTE:
        results[key] = published_sweeps()['results'][key]
        return results[key], f'published:sweeps_{GROUP}:{key}'
    # Config conversion and optimization live only inside the explicit producer.
    def produce():
        cases = prepare_study_cases(key) if prepared_cases is None else prepared_cases
        return optimize_parameter_sweep(theory, cases, R_VALUES, **SEARCH)
    curves, identity = ResultCache(**CACHE_OPTIONS).get('reproduction_sweep_theory',
        study_inputs(key), produce, sources=THEORY_SOURCES,
        implementation=[inspect.getsource(study_inputs), inspect.getsource(prepare_study_cases),
                        inspect.getsource(load_study)], compute=compute, force=force)
    results[key] = curves
    return curves, identity


def load_study_evidence(key, *, compute=False, force=False):
    if not RECOMPUTE:
        TRANSITIONS[key] = published_sweeps()['transitions'][key]
        return TRANSITIONS[key]
    curves, _ = load_study(key, compute=False)
    options = TRANSITION_OPTIONS | STUDY_TRANSITION_OPTIONS.get(key, {})
    cache = ResultCache(**CACHE_OPTIONS)
    if compute:
        entry = cached_sweep_transitions(curves, options, cache, force=force)
    else:
        signature = sweep_transition_signature(curves, options)
        evidence, identity = cache.get('sweep_transitions',
            dict(curves_sha256=signature, options=options), sources=TRANSITION_SOURCES,
            compute=False)
        entry = dict(signature=signature, evidence=evidence, identity=identity)
    TRANSITIONS[key] = entry
    return entry


def stored_sweep_transitions(key, curves):
    options = TRANSITION_OPTIONS | STUDY_TRANSITION_OPTIONS.get(key, {})
    entry = TRANSITIONS.get(key)
    if entry is None or entry['signature'] != sweep_transition_signature(curves, options):
        raise ValueError(f"{key}: missing/stale barrier evidence; run this study's diagnostic cell. "
                         "Plotting does not evaluate risk.")
    return entry['evidence']


## Optimized curves

Validate the selected inputs and load the included curves. With `RECOMPUTE=True`, this stage runs the ridge searches.


In [ ]:
prepared, errors = {}, []
for key in APPENDIX_GROUPS[GROUP]:
    try:
        prepared[key] = prepare_study_cases(key)
    except ValueError as exc:
        errors.append(f'{key}: {exc}')
if errors:
    raise ValueError('Infeasible sweep configurations:\n' + '\n'.join(errors))
for key in APPENDIX_GROUPS[GROUP]:
    curves, _ = load_study(key, compute=True, force=FORCE_THEORY, prepared_cases=prepared[key])
    print(key, 'values:', [c.case.value for c in curves],
          'finite-cap hits:', sum(int(np.sum(c.optimum.boundary == 'upper')) for c in curves))


## Diagnose barriers separately

For each pair of neighboring aspect-ratio samples, the same risk-barrier check
used for landscapes probes between the selected penalties at **both** endpoints.
Dashed connectors require resolved barriers at both endpoints. Failure to find a
barrier is inconclusive; a steep curve or a finite-cap hit is not evidence of a
jump or of infinite ridge. This stage evaluates risk, but never changes optima.

In [ ]:
for key in APPENDIX_GROUPS[GROUP]:
    entry = load_study_evidence(key, compute=True, force=FORCE_DIAGNOSTICS)
    print(key, 'barrier-supported intervals:', [int(e['jump_mask'].sum()) for e in entry['evidence']])


## Editable figure construction
These helpers use stored arrays and diagnostics only. Row limits are resolved before clipping paths to zero.

In [ ]:
def ridge_jump_path(r, values, *, style, jump_mask):
    """Break only diagnosed intervals and connect their actual sampled endpoints."""
    if style not in ("dashed", "solid", "gap"):
        raise ValueError("jump_style must be dashed, solid, or gap")
    r, values = np.asarray(r), np.asarray(values)
    jump_mask = np.asarray(jump_mask, bool)
    if jump_mask.shape != (len(r)-1,):
        raise ValueError("Barrier mask must match the sampled r intervals")
    if style == "solid":
        return r, values, []
    xs, ys, connectors = [r[0]], [values[0]], []
    for j, jump in enumerate(jump_mask):
        if jump:
            xs.append(np.nan)
            ys.append(np.nan)
            if style == "dashed":
                connectors.append((r[j:j+2].copy(), values[j:j+2].copy()))
        xs.append(r[j+1])
        ys.append(values[j+1])
    return np.asarray(xs), np.asarray(ys), connectors

def plot_parameter_pair(key, curves, *, options=None, axes=None, show=True):
    spec = SWEEPS[key]
    opts = PLOT | STUDY_PLOT.get(key, {}) | (options or {})
    strip = opts["zero_display"] == "strip"
    if opts["zero_display"] not in ("strip", "axis"):
        raise ValueError('zero_display must be "strip" or "axis"')
    if not strip and opts["ridge_scale"] == "log":
        raise ValueError('Use ridge_scale="symlog" or "linear" for zero_display="axis"')
    if not curves:
        raise ValueError("No curves to plot")
    evidence = stored_sweep_transitions(key, curves) if opts["jump_style"] != "solid" else None
    # Use a continuous palette instead of recycling colors after four curves.
    from matplotlib.colors import to_rgba
    anchors = np.asarray([to_rgba(color) for color in opts["palette"]])
    if len(anchors) < 2:
        raise ValueError("palette needs at least two gradient anchor colors")
    positions = np.linspace(0, 1, len(curves))
    anchor_positions = np.linspace(0, 1, len(anchors))
    curve_colors = np.column_stack([
        np.interp(positions, anchor_positions, anchors[:, channel])
        for channel in range(4)
    ])
    if axes is None:
        fig = plt.figure(figsize=(opts["figure_width"], opts["figure_height"]), dpi=opts["dpi"], layout="constrained")
        if strip:
            gs = fig.add_gridspec(2, 2, height_ratios=[1, opts["zero_strip_height"]])
            left = fig.add_subplot(gs[0, 0])
            zeros_ax = fig.add_subplot(gs[1, 0], sharex=left)
            right = fig.add_subplot(gs[:, 1])
            left.tick_params(axis="x", labelbottom=False)
        else:
            left, right = fig.subplots(1, 2)
            zeros_ax = None
    else:
        left, right, zeros_ax = axes
        fig = left.figure
        if (zeros_ax is not None) != strip:
            raise ValueError('Provided axes must match zero_display')
        if strip:
            left.tick_params(axis='x', labelbottom=False)
    coordinate = opts["ridge_coordinate"]
    normalization = opts["risk_normalization"]
    left_label = r"Optimal ridge $\lambda_{\text{opt}}$" if coordinate == "lambda" else r"Effective $\lambda_{\text{opt}}$"
    right_labels = {"R_infinity": r"Optimal risk $R_{\text{opt}}/R_\infty$", "sigma2": r"$R_{\text{opt}}/\sigma^2$", "none": r"$R_{\text{opt}}$"}
    labels, colors = [], []
    ridge_lines = []  # restore zero endpoints after fixing positive-only y limits
    any_upper = False
    for i, curve in enumerate(curves):
        color = curve_colors[i]
        label = opts["curve_labels"].get(curve.case.value, fr"${spec['symbol']}={curve.case.value:g}$")
        if opts["mark_base"] and np.isclose(curve.case.value, (REFERENCE_L if spec["parameter"] == "L" else
                    BASE.FTest - BASE.F + BASE.Fneq if spec["parameter"] == "Fneqtilde" else
                    getattr(BASE, spec["parameter"]))):
            label += " (base)"
        labels.append(label)
        colors.append(color)
        r = curve.optimum.r_values
        ridge = curve.regularization_values(coordinate)
        risk = curve.risk_values(normalization)
        zero = curve.optimum.boundary == "zero"
        if not hasattr(curve.optimum, "zero_slopes"):
            raise ValueError("Cached results use the old lower-bound search. Recompute with SEARCH method='nonnegative'.")
        marker = opts["zero_markers"][i % len(opts["zero_markers"])]
        for ax, y, scale, limits in ((left, ridge, opts["ridge_scale"], opts["ridge_ylim"]),
                                     (right, risk, opts["risk_scale"], opts["risk_ylim"])):
            if ax is left:
                mask = evidence[i]['jump_mask'].copy() if evidence is not None else np.zeros(len(r)-1, bool)
                manual = opts['jump_intervals_by_value'].get(curve.case.value)
                if manual is not None:
                    indices = np.asarray(manual, dtype=int)
                    if np.any(indices < 0) or np.any(indices >= len(mask)):
                        raise ValueError('Manual jump interval outside the sampled grid')
                    mask[:] = False
                    mask[indices] = True
                plot_r, plot_y, jumps = ridge_jump_path(
                    r, y, style=opts["jump_style"], jump_mask=mask)
            else:
                plot_r, plot_y, jumps = r, y, []
            shown = np.where(plot_y == 0, np.nan, plot_y) if ax is left and strip else plot_y
            finite = shown[np.isfinite(shown)]
            if scale == "log" and np.any(finite <= 0):
                raise ValueError(f"{key}: non-positive values cannot be shown on a log axis")
            line, = ax.plot(plot_r, shown, color=color, linestyle="-", lw=opts["linewidth"], marker=opts["point_marker"],
                            ms=opts["point_size"], alpha=opts["alpha"], label=label)
            if ax is left and strip:
                ridge_lines.append((line, plot_y))
            for jump_r, jump_y in jumps:
                shown_jump = np.where(jump_y == 0, np.nan, jump_y) if strip else jump_y
                connector, = ax.plot(jump_r, shown_jump, linestyle="--",
                                    color=color, lw=opts["linewidth"], alpha=opts["alpha"])
                if strip:
                    ridge_lines.append((connector, jump_y))
            upper = curve.optimum.boundary == "upper"
            any_upper |= bool(np.any(upper))
            if opts["show_upper"]:
                ax.plot(r[upper], y[upper], linestyle="none", marker=opts["upper_marker"],
                        ms=opts["upper_size"], color=color, markerfacecolor="none")
            if np.any(zero) and ((ax is left and not strip) or (ax is right and opts["mark_zero_on_risk"])):
                ax.plot(r[zero], y[zero], linestyle="none", marker=marker,
                        ms=opts["zero_size"], color=color,
                        markerfacecolor=color if opts["zero_filled"] else "none")
            if limits is not None and finite.size:
                lo, hi = limits
                if (lo is not None and finite.min() < lo) or (hi is not None and finite.max() > hi):
                    print(f"WARNING: {key}, value={curve.case.value:g}: explicit y limits clip plotted values")
        if strip:
            # NaN gaps prevent joining disjoint ridgeless intervals.
            zeros_ax.plot(r, np.where(zero, float(i), np.nan), linestyle="-", marker=marker,
                          lw=opts["linewidth"], ms=opts["zero_size"], color=color,
                          solid_capstyle="butt",
                          markerfacecolor=color if opts["zero_filled"] else "none")
            # An isolated zero sample has no line segment: show a plain short dash.
            isolated = zero & ~np.r_[False, zero[:-1]] & ~np.r_[zero[1:], False]
            if not marker and np.any(isolated):
                zeros_ax.plot(r[isolated], np.full(np.sum(isolated), i), linestyle="none",
                              marker="_", ms=opts["zero_size"],
                              markeredgewidth=opts["linewidth"], color=color)
    if normalization == "R_infinity" and opts["show_rinf"]:
        right.axhline(1, color="0.65", ls=":", lw=1, zorder=0)
    for ax, title, ylabel, scale, limits in (
        (left, opts["ridge_title"], opts["ridge_label"] or left_label, opts["ridge_scale"], opts["ridge_ylim"]),
        (right, opts["risk_title"], opts["risk_label"] or right_labels[normalization], opts["risk_scale"], opts["risk_ylim"]),
    ):
        ax.set(title=title, ylabel=ylabel, xscale=opts["x_scale"])
        if scale == "symlog":
            ax.set_yscale(scale, linthresh=opts["symlog_linthresh"])
        else:
            ax.set_yscale(scale)
        if limits is not None:
            ax.set_ylim(limits)
        if opts["xlim"] is not None:
            ax.set_xlim(opts["xlim"])
        if opts["show_legend"] and ax is right:
            ax.legend(loc=opts["legend_loc"], fontsize=opts["legend_fontsize"], frameon=True)
        if opts["grid"]:
            ax.grid(alpha=opts["grid_alpha"])
    if strip:
        # Autoscaling above saw only positive ridge values. Freeze those limits
        # (including any explicit override), then continue each line to true zero.
        # Log axes clip zero below the panel, extending transitions to its edge.
        left.set_ylim(left.get_ylim())
        if opts["ridge_scale"] == "log":
            left.set_yscale("log", nonpositive="clip")
        for line, ridge_values in ridge_lines:
            line.set_ydata(ridge_values)
        # Do not relim/autoscale after restoring zero endpoints.
    right.set_xlabel(opts["x_label"])
    if strip:
        zeros_ax.set(yticks=np.arange(len(curves)) if opts["zero_show_curve_labels"] else [],
                     yticklabels=labels if opts["zero_show_curve_labels"] else [],
                     ylim=(len(curves)-0.5, -0.5), xlabel=opts["x_label"],
                     ylabel=opts["zero_label"])
        zeros_ax.set_ylabel(opts["zero_label"], rotation=opts["zero_label_rotation"],
                            labelpad=opts["zero_label_pad"], ha="right", va="center")
        zeros_ax.tick_params(axis="y", length=0, labelsize=opts["zero_label_size"])
        for tick, color in zip(zeros_ax.get_yticklabels(), colors):
            tick.set_color(color)
        zeros_ax.spines[["top", "right", "left"]].set_visible(False)
        if opts["grid"]:
            zeros_ax.grid(axis="x", alpha=opts["grid_alpha"])
    else:
        left.set_xlabel(opts["x_label"])
    # Resolve defaults at plot time so edits to first-cell rcParams remain effective.
    text_sizes = {
        name: plt.rcParams[rc_key] if opts[name] is None else opts[name]
        for name, rc_key in (
            ("x_label_size", "axes.labelsize"), ("y_label_size", "axes.labelsize"),
            ("x_tick_size", "xtick.labelsize"), ("y_tick_size", "ytick.labelsize"),
        )
    }
    for ax in (left, right, zeros_ax) if strip else (left, right):
        ax.xaxis.label.set_fontsize(text_sizes["x_label_size"])
        ax.yaxis.label.set_fontsize(text_sizes["y_label_size"])
        ax.tick_params(axis="x", which="both", labelsize=text_sizes["x_tick_size"])
        if ax is not zeros_ax:
            ax.tick_params(axis="y", which="both", labelsize=text_sizes["y_tick_size"])

    title = spec["title"] if opts["figure_title"] is None else opts["figure_title"]
    if opts["show_constraints"]:
        title += "\n" + REGIMES[spec["regime"]]["caption"]
    if title and axes is None:
        fig.suptitle(title, fontsize=opts["title_size"])
    if any_upper:
        print("Upper-bound optima found: consider increasing SEARCH['lambda_max'].")
    if show:
        plt.show()
    return fig

def appendix_row_limits(series, scale, explicit=None):
    """Presentation limits only, from stored displayed arrays."""
    values = np.concatenate([np.asarray(v, float).ravel() for v in series])
    values = values[np.isfinite(values)]
    if scale == 'log':
        values = values[values > 0]
    if not values.size:
        low, high = (1e-6, 1.) if scale == 'log' else (0., 1.)
    else:
        low, high = float(values.min()), float(values.max())
        if scale == 'log':
            pad = .05 * max(np.log(high / low), 1.)
            low, high = low * np.exp(-pad), high * np.exp(pad)
        else:
            pad = .05 * max(high - low, 1e-6 * max(abs(low), 1.))
            low, high = low - pad, high + pad
    if explicit is not None:
        low = low if explicit[0] is None else explicit[0]
        high = high if explicit[1] is None else explicit[1]
    if not np.all(np.isfinite([low, high])) or low >= high or (scale == 'log' and low <= 0):
        raise ValueError('Row limits must be finite/increasing and positive for a log axis')
    return low, high


def appendix_sweep_inputs(keys):
    """Check selected stored results without preparing configs or evaluating theory."""
    missing = [key for key in keys if not results.get(key)]
    if missing:
        raise KeyError('Missing stored sweep results: ' + ', '.join(missing)
                       + '. Run only their explicit study cells; this plot never computes them.')
    selected = {}
    for key in keys:
        spec = SWEEPS[key]
        curves = results[key]
        if [float(c.case.value) for c in curves] != list(map(float, spec['values'])):
            raise ValueError(f'{key}: stored values differ from SWEEPS; restore matching settings/results')
        # Compare stored input metadata to the current controlled path; no theory calls.
        regime = REGIMES[spec['regime']]
        for curve in curves:
            expected = asdict(BASE) | regime['fixed']
            if spec.get('path') == 'differing_signals':
                train, test = ((float(curve.case.value), spec['fixed_other'])
                               if spec['parameter'] == 'Fneq'
                               else (spec['fixed_other'], float(curve.case.value)))
                expected.update(Fneq=train, FTest=BASE.F-train+test)
            elif spec['parameter'] != 'L':
                expected[spec['parameter']] = float(curve.case.value)
            changed = set(regime['fixed']) | {spec['parameter']}
            if 'rhoneq' in changed and 'Cneq' not in changed:
                expected['Cneq'] = None
            elif 'Cneq' in changed and 'rhoneq' not in changed:
                expected['rhoneq'] = None
            expected.update({target: expected[source] for target, source in regime['ties'].items()})
            length = int(curve.case.value) if spec['parameter'] == 'L' else REFERENCE_L
            if (asdict(curve.case.parameters) != expected or curve.case.reference_L != length
                    or curve.case.parameter != spec['parameter']
                    or not np.array_equal(curve.optimum.r_values, R_VALUES)):
                raise ValueError(f'{key}: stored inputs differ from current controls; restore matching results')
        selected[key] = curves
    return selected


def draw_appendix_sweeps(keys, selected, style, column_options):
    from matplotlib.ticker import FixedLocator
    n = len(keys)
    if not n or len(set(keys)) != n:
        raise ValueError('Select distinct study keys for the appendix group')
    opts = [dict(PLOT | STUDY_PLOT.get(key, {}) | style['plot'] | column_options.get(key, {}))
            for key in keys]
    for field in ('ridge_coordinate', 'risk_normalization', 'x_scale', 'ridge_scale', 'risk_scale'):
        if len({o[field] for o in opts}) != 1:
            raise ValueError(f'All columns need the same {field}; use separate groups otherwise')
    if any(o['zero_display'] != 'strip' for o in opts):
        raise ValueError('Grouped figures use zero_display="strip"; use individual views for axis mode')
    if style['share_y']:
        ridge_arrays = [c.regularization_values(opts[0]['ridge_coordinate'])
                        for key in keys for c in selected[key]]
        risk_arrays = [c.risk_values(opts[0]['risk_normalization'])
                       for key in keys for c in selected[key]]
        if opts[0]['risk_normalization'] == 'R_infinity' and any(o['show_rinf'] for o in opts):
            risk_arrays.append(np.array([1.]))
        common_ridge = appendix_row_limits(ridge_arrays, opts[0]['ridge_scale'], style['ridge_ylim'])
        common_risk = appendix_row_limits(risk_arrays, opts[0]['risk_scale'], style['risk_ylim'])
        for o in opts:
            o.update(ridge_ylim=common_ridge, risk_ylim=common_risk)
    fig = plt.figure(figsize=(style['width'], style['height']), dpi=style['dpi'], layout='constrained')
    fig.get_layout_engine().set(**style['layout_padding'])
    gs = fig.add_gridspec(3, n, height_ratios=[1., style['zero_strip_height'], 1.])
    ridge_axes, risk_axes = [], []
    for col, (key, o) in enumerate(zip(keys, opts)):
        top = fig.add_subplot(gs[0, col])
        zero = fig.add_subplot(gs[1, col], sharex=top)
        bottom = fig.add_subplot(gs[2, col], sharex=top)
        o.update(figure_title='', show_constraints=False, ridge_title='', risk_title='')
        plot_parameter_pair(key, selected[key], options=o, axes=(top, bottom, zero), show=False)
        zero.set_xlabel('')
        zero.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        top.set_title(style['titles'].get(key, SWEEPS[key]['title']), fontsize=style['title_size'])
        if style['share_y'] and col:
            for ax in (top, bottom):
                ax.set_ylabel('')
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
            zero.set_ylabel('')
        ridge_axes.append(top)
        risk_axes.append(bottom)
        if style['panel_labels']:
            for ax, letter in ((top, chr(97 + col)), (bottom, chr(97 + n + col))):
                ax.text(*style['panel_label_xy'], letter, transform=ax.transAxes,
                        fontsize=style['title_size'], fontweight='bold')
    # Common limits were resolved before zero-boundary paths were drawn.
    for axes, ticks in ((ridge_axes, style['ridge_ticks']), (risk_axes, style['risk_ticks'])):
        if style['share_y'] and ticks is None:
            ticks = axes[0].get_yticks()
        if ticks is not None:
            ticks = np.asarray(ticks, float)
            if ticks.ndim != 1 or np.any(~np.isfinite(ticks)) or np.any(np.diff(ticks) <= 0):
                raise ValueError('Row ticks must be finite and increasing')
            for ax in axes:
                ax.yaxis.set_major_locator(FixedLocator(ticks))
    return fig


def plot_appendix_sweeps(group, *, force=False):
    """Cached plot-only assembly from stored curves, with live renderer fingerprints."""
    import inspect
    import pickle
    from hashlib import sha256
    from dms.result_cache import ResultCache
    keys = APPENDIX_GROUPS[group]
    selected = appendix_sweep_inputs(keys)
    evidence = {key: stored_sweep_transitions(key, selected[key]) for key in keys}
    overrides = APPENDIX_GROUP_STYLE.get(group, {})
    style = dict(APPENDIX_STYLE | overrides)
    for nested in ('plot', 'titles', 'layout_padding'):
        style[nested] = APPENDIX_STYLE[nested] | overrides.get(nested, {})
    columns = APPENDIX_COLUMN_STYLE.get(group, {})
    inputs = dict(group=group, keys=keys,
                  evidence_sha256=sha256(pickle.dumps(evidence)).hexdigest(), data_sha256=sha256(pickle.dumps(selected)).hexdigest(),
                  specs={key: SWEEPS[key] for key in keys}, baseline=asdict(BASE), reference_L=REFERENCE_L,
                  plot=PLOT, study_plot={key:STUDY_PLOT.get(key, {}) for key in keys},
                  style=style, columns=columns,
                  rcparams={k:repr(v) for k,v in plt.rcParams.items()})
    helpers = (ridge_jump_path, stored_sweep_transitions, plot_parameter_pair, appendix_row_limits,
               appendix_sweep_inputs, draw_appendix_sweeps, plot_appendix_sweeps)
    try:
        implementation = [inspect.getsource(f) for f in helpers]
    except (OSError, TypeError):
        implementation, force = [], True
    fig, _ = ResultCache(**APPENDIX_CACHE_OPTIONS).get('appendix_sweep_figure', inputs,
        lambda: draw_appendix_sweeps(keys, selected, style, columns),
        sources=[PROJECT_ROOT/'dms'/'exploration.py'], implementation=implementation, force=force)
    APPENDIX_FIGURES['appendix_' + group] = fig
    display(fig)
    plt.close(fig)
    return fig


In [ ]:
PLOT = dict(
    # What to show
    ridge_coordinate="lambda",           # "lambda": Ridge lambda; "mu": Effective lambda
    risk_normalization="R_infinity",      # "R_infinity", "sigma2", "none"
    zero_display="strip",                # "strip" (recommended) or "axis"

    # Axis scales and limits
    x_scale="log", ridge_scale="log", risk_scale="linear",
    xlim=None,                           # e.g. (0.01, 100)
    ridge_ylim=None,                     # e.g. (1e-5, 1)
    risk_ylim=None,                      # e.g. (0, 1.05)
    symlog_linthresh=1e-5,               # only used with a symlog y axis

    # Ridgeless strip: plain colored lines; identify curves using the legend
    zero_markers=[""],                   # optional markers; none by default
    zero_show_curve_labels=False,        # legend already identifies each color
    zero_size=4.0, zero_filled=True,
    zero_strip_height=0.16,              # height relative to positive-ridge panel
    zero_label=r"$\lambda_{opt}=0$",
    zero_label_rotation=0,               # horizontal (90 would be vertical)
    zero_label_pad=5,
    zero_label_size=11,
    mark_zero_on_risk=False,

    # Barrier-supported connectors (stored diagnostics; no evaluations while plotting)
    jump_style="dashed",                # "dashed", "solid" (original), or "gap"
    jump_intervals_by_value={},         # optional manual interval-index overrides per curve value

    # Curves and boundary markers
    # Gradient anchors: automatically resampled to the number of curves.
    palette=["#243247", "#765033", "#b46d2b", "#ed932e"],
    linewidth=1.6, alpha=0.95,
    point_marker="", point_size=2.0,     # "" hides ordinary sample markers
    show_upper=True, upper_marker="^", upper_size=5.0,

    # Text sizes: None inherits rcParams from the first cell (currently 11 pt).
    # These also apply to the ridgeless strip. Set numbers to override.
    x_label_size=None, y_label_size=None,
    x_tick_size=None, y_tick_size=None,

    # Labels and titles (None selects the automatic label/title)
    x_label=r"Aspect ratio $r=qL/n$",
    ridge_label=None, risk_label=None,
    ridge_title="",#"Optimal regularization", 
    risk_title="",#"Optimal prediction risk",
    figure_title=None, show_constraints=True, title_size=12,
    curve_labels={}, mark_base=True,

    # Layout, legend, and guides
    figure_width=10.0, figure_height=4,  # inches, for the whole figure
    dpi=140,                            # display resolution
    show_legend=True, legend_loc="best", legend_fontsize=11,
    grid=False, grid_alpha=0.18, show_rinf=True,
)

# Optional per-study overrides; keys match SWEEPS. All PLOT keys are supported.
STUDY_PLOT = {
    # "G2": dict(ridge_ylim=(1e-5, 1), risk_ylim=(0, 1.05)),
    # "F": dict(curve_labels={0.4: r"$F=0.4$"}, mark_base=False),
}

APPENDIX_STYLE = dict(
    width=10., height=6., dpi=140,
    share_y=True, ridge_ylim=None, risk_ylim=None, ridge_ticks=None, risk_ticks=None,
    zero_strip_height=.13,
    title_size=12, panel_labels=True, panel_label_xy=(-.10, 1.03),
    layout_padding=dict(w_pad=.03, h_pad=.03, wspace=.04, hspace=.03),
    titles={'G2': r'Signal strength $G^2$', 'F': r'Matched signal $F=\tilde F$',
            'f_common': r'Common breadth $f=\tilde f$', 'f_train': r'Training breadth $f$',
            'f_test': r'Test breadth $\tilde f$', 'FTest': r'Test signal $\tilde F$',
            'Fneq': r'Differing-site signal $F_{\neq}$',
            'Fneq_independent': r'Training signal $F_{\neq}$',
            'Fneqtilde_independent': r'Test signal $\tilde F_{\neq}$', 'm': r'Total mean $m$',
            'mneq': r'Training mean $m_{\neq}$', 'mneqtilde': r'Test mean $\tilde m_{\neq}$'},
    plot=dict(risk_normalization='R_infinity', ridge_coordinate='lambda',
              zero_display='strip', mark_base=False,  # inherit each study's jump settings
              x_label_size=11, y_label_size=11, x_tick_size=11, y_tick_size=11,
              legend_fontsize=11, show_constraints=False, figure_title=''),
)
APPENDIX_GROUP_STYLE = {
    'energies': dict(width=7.2),
    # 'means': dict(risk_ylim=(0., 1.05), risk_ticks=[0., .25, .5, .75, 1.]),
}
APPENDIX_COLUMN_STYLE = {
    # 'matched': {'F': dict(legend_loc='lower right')},
}

APPENDIX_CACHE_OPTIONS = dict(CACHE_OPTIONS)
FORCE_REDRAW = False


## Draw from the loaded results

The plot uses the selected curves and barrier evidence, with no numerical calculations.


In [ ]:
for key in APPENDIX_GROUPS[GROUP]:
    load_study(key, compute=False)
    load_study_evidence(key, compute=False)
figure = plot_appendix_sweeps(GROUP, force=FORCE_REDRAW)


## Export displayed PDFs
Only the selected displayed group is exported by default. Edit `PDF_FIGURES` to include additional groups you have already plotted in this kernel.

In [ ]:
PDF_NAMES = {
    'matched':'appendix_sweeps_matched.pdf',
    'energies':'appendix_sweeps_energies.pdf',
    'means':'appendix_sweeps_means.pdf',
}
PDF_FIGURES = {'appendix_'+GROUP:PDF_NAMES[GROUP]}
missing = [key for key in PDF_FIGURES if key not in APPENDIX_FIGURES]
if missing:
    raise KeyError(f'Display the selected figures first: {missing}; no calculations started.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key, filename in PDF_FIGURES.items():
    output = OUTPUT_DIR/Path(filename).with_suffix('.pdf')
    APPENDIX_FIGURES[key].savefig(output, format='pdf', bbox_inches='tight')
    print(output)
